<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_24_Stateful_AI_Conversations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🚀 Day 23 — Stateful AI Conversations

## Session-Aware AI Assistant with Conversation Memory

### 🎯 Focus Area

Large Language Models are generally **stateless**. Each independent API/model call does not automatically remember previous conversation turns. This makes follow-up questions such as *"What did you mean by that?"* difficult to answer without providing previous context.

This project implements a **stateful AI conversation system** that maintains conversation memory across multiple turns while controlling context size to avoid unnecessary token and computational overhead.

---

## 🧠 Project Objective

Build a session-aware AI assistant that:

- Maintains conversation history for each user session.
- Stores messages as `role-content` pairs.
- Provides `append()`, `get_context()`, and `clear()` methods.
- Uses only the **latest five conversation turns** as active context.
- Summarizes the oldest five turns when a conversation exceeds ten turns.
- Compares responses **with and without memory**.
- Measures the token overhead introduced by conversation memory.
- Provides a FastAPI endpoint for session-aware conversations.

---

## 🛠️ Technologies Used

| Technology | Purpose |
|---|---|
| Python | Core implementation |
| FastAPI | REST API |
| Hugging Face Transformers | Local LLM |
| PyTorch | Model inference |
| Qwen2.5-0.5B-Instruct | Local language model |
| LangChain concepts | Conversation and LLM workflow |
| Google Colab | Development environment |

> **Note:** This implementation uses a local Hugging Face model and does **not require an OpenAI API key**.

---

## 🏗️ System Architecture

```text
                    ┌──────────────────┐
                    │       USER       │
                    └────────┬─────────┘
                             │
                             ▼
                    ┌──────────────────┐
                    │     FastAPI      │
                    │     /chat        │
                    └────────┬─────────┘
                             │
                         session_id
                             │
                             ▼
                    ┌──────────────────┐
                    │ Session Manager  │
                    │   Dictionary     │
                    └────────┬─────────┘
                             │
                             ▼
                ┌──────────────────────────┐
                │   ConversationHistory   │
                │                          │
                │   append()              │
                │   get_context()         │
                │   clear()               │
                └────────────┬─────────────┘
                             │
                       Last 5 Turns
                             │
                             ▼
                    ┌──────────────────┐
                    │ Context Injection│
                    └────────┬─────────┘
                             │
                             ▼
                    ┌──────────────────┐
                    │  Local Qwen LLM │
                    └────────┬─────────┘
                             │
                             ▼
                    ┌──────────────────┐
                    │   AI Response    │
                    └──────────────────┘

In [1]:
# ============================================================
# DAY 23 — STATEFUL AI CONVERSATIONS
# Local Hugging Face LLM + Conversation Memory + FastAPI
# No OpenAI API required
# ============================================================

# -----------------------------
# 1. Install dependencies
# -----------------------------
!pip -q install transformers accelerate fastapi uvicorn nest-asyncio requests


# -----------------------------
# 2. Imports
# -----------------------------
import torch
import uuid
import threading
import nest_asyncio
import requests

from typing import Optional
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastapi import FastAPI
from pydantic import BaseModel

nest_asyncio.apply()


# ============================================================
# 3. Check GPU
# ============================================================

print("=" * 60)
print("GPU CHECK")
print("=" * 60)

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


# ============================================================
# 4. Load Local Hugging Face Model
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("\nLoading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

print("Model loaded successfully!")


# ============================================================
# 5. Local LLM Function
# ============================================================

def generate_response(prompt, max_new_tokens=150):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return response.strip()


# ============================================================
# 6. ConversationHistory Class
# ============================================================

class ConversationHistory:

    def __init__(self):
        self.messages = []

    def append(self, role, content):

        self.messages.append({
            "role": role,
            "content": content
        })

    def get_context(self, max_turns=5):

        # One turn = user + assistant
        max_messages = max_turns * 2

        return self.messages[-max_messages:]

    def clear(self):

        self.messages = []

    def __len__(self):

        return len(self.messages)


# ============================================================
# 7. Session Memory
# ============================================================

sessions = {}


def get_session(session_id):

    if session_id not in sessions:

        sessions[session_id] = ConversationHistory()

    return sessions[session_id]


# ============================================================
# 8. Context Injection
# ============================================================

def build_prompt(history, user_message):

    context = history.get_context(
        max_turns=5
    )

    context_text = ""

    for message in context:

        context_text += (
            f"{message['role'].upper()}: "
            f"{message['content']}\n"
        )

    prompt = f"""
You are a helpful AI assistant.

Use the conversation history to answer
the current question.

CONVERSATION HISTORY:
{context_text}

CURRENT USER MESSAGE:
USER: {user_message}

Answer naturally and use previous context
whenever it is relevant.
"""

    return prompt


# ============================================================
# 9. Memory-Based Chat
# ============================================================

def chat_with_memory(session_id, user_message):

    history = get_session(session_id)

    prompt = build_prompt(
        history,
        user_message
    )

    response = generate_response(prompt)

    # Store user message
    history.append(
        "user",
        user_message
    )

    # Store AI response
    history.append(
        "assistant",
        response
    )

    # Memory truncation
    compress_memory(history)

    return response


# ============================================================
# 10. Memory Truncation / Summarization
# ============================================================

MAX_TURNS = 10
SUMMARY_TURNS = 5


def summarize_old_messages(messages):

    conversation = ""

    for message in messages:

        conversation += (
            f"{message['role'].upper()}: "
            f"{message['content']}\n"
        )

    prompt = f"""
Summarize the following conversation for
future memory.

Preserve important:
- names
- facts
- preferences
- projects
- decisions
- important context

Do not invent information.

CONVERSATION:
{conversation}

Return only a concise summary.
"""

    return generate_response(
        prompt,
        max_new_tokens=150
    )


def compress_memory(history):

    turn_count = len(history.messages) // 2

    if turn_count <= MAX_TURNS:
        return

    # Oldest five turns = 10 messages
    oldest_messages = history.messages[:10]

    summary = summarize_old_messages(
        oldest_messages
    )

    remaining_messages = history.messages[10:]

    history.messages = [
        {
            "role": "system",
            "content":
                "Previous conversation summary:\n"
                + summary
        }
    ] + remaining_messages


# ============================================================
# 11. Chat WITHOUT Memory
# ============================================================

def chat_without_memory(message):

    prompt = f"""
You are a helpful AI assistant.

Answer the user's question independently.

USER:
{message}
"""

    return generate_response(prompt)


# ============================================================
# 12. Five-Turn Memory Test
# ============================================================

print("\n")
print("=" * 60)
print("FIVE-TURN MEMORY TEST")
print("=" * 60)

test_messages = [

    "My name is Radhika.",

    "I am building a healthcare AI assistant using Python.",

    "What am I building?",

    "It should help patients understand their symptoms.",

    "What should it help them understand?"
]

test_session = "five-turn-demo"

for turn, message in enumerate(
    test_messages,
    start=1
):

    answer = chat_with_memory(
        test_session,
        message
    )

    print("\n" + "-" * 60)
    print("TURN:", turn)
    print("USER:", message)
    print("AI:", answer)


# ============================================================
# 13. Compare WITHOUT Memory
# ============================================================

print("\n")
print("=" * 60)
print("WITHOUT MEMORY TEST")
print("=" * 60)

for turn, message in enumerate(
    test_messages,
    start=1
):

    answer = chat_without_memory(
        message
    )

    print("\n" + "-" * 60)
    print("TURN:", turn)
    print("USER:", message)
    print("AI:", answer)


# ============================================================
# 14. FastAPI Application
# ============================================================

app = FastAPI(
    title="Stateful AI Conversation API",
    version="1.0"
)


# ============================================================
# 15. API Request Model
# ============================================================

class ChatRequest(BaseModel):

    message: str

    session_id: Optional[str] = None


# ============================================================
# 16. Chat Endpoint
# ============================================================

@app.post("/chat")
def chat(request: ChatRequest):

    session_id = request.session_id

    if not session_id:

        session_id = str(uuid.uuid4())

    response = chat_with_memory(
        session_id,
        request.message
    )

    history = get_session(
        session_id
    )

    return {
        "session_id": session_id,
        "response": response,
        "stored_messages": len(history.messages)
    }


# ============================================================
# 17. Clear Session Endpoint
# ============================================================

@app.delete("/session/{session_id}")
def clear_session(session_id: str):

    if session_id in sessions:

        sessions[session_id].clear()

        return {
            "status": "success",
            "message": "Memory cleared"
        }

    return {
        "status": "not_found",
        "message": "Session not found"
    }


# ============================================================
# 18. Start FastAPI Server
# ============================================================

def run_server():

    import uvicorn

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()


# ============================================================
# 19. Final Status
# ============================================================

print("\n")
print("=" * 60)
print("PROJECT READY")
print("=" * 60)

print("""
✓ Local Hugging Face LLM
✓ No OpenAI API
✓ ConversationHistory class
✓ append()
✓ get_context()
✓ clear()
✓ Session-based memory
✓ Last 5 turns context window
✓ 10-turn memory limit
✓ Oldest 5 turns summarization
✓ With-memory test
✓ Without-memory test
✓ FastAPI /chat endpoint
✓ DELETE /session/{session_id}
""")

print("FastAPI server: http://127.0.0.1:8000")

GPU CHECK
CUDA Available: False
Running on CPU

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


FIVE-TURN MEMORY TEST

------------------------------------------------------------
TURN: 1
USER: My name is Radhika.
AI: Hello Radhika! How can I assist you today?

------------------------------------------------------------
TURN: 2
USER: I am building a healthcare AI assistant using Python.
AI: Hello Radhika! That sounds like a great initiative for your healthcare AI assistant project. Building an AI assistant can be both exciting and challenging. What specific aspects of AI health care do you have in mind? Do you have any particular challenges or goals you're aiming to achieve with this project?

------------------------------------------------------------
TURN: 3
USER: What am I building?
AI: Hey Radhika! It's nice to hear from you. I'm building a healthcare AI assistant using Python. The main idea is to create a tool that helps doctors and patients better understand their health conditions and provide personalized treatment plans. Specifically, I want

In [2]:
# ============================================================
# INTERACTIVE SESSION-AWARE CHAT
# ============================================================

session_id = "radhika-demo"

print("Session ID:", session_id)
print("Type 'clear' to erase memory.")
print("Type 'exit' to stop.\n")

while True:

    user_message = input("You: ")

    if user_message.lower() == "exit":
        print("Chat ended.")
        break

    if user_message.lower() == "clear":

        if session_id in sessions:
            sessions[session_id].clear()

        print("✓ Memory cleared.\n")
        continue

    response = chat_with_memory(
        session_id,
        user_message
    )

    print("AI:", response)
    print()

Session ID: radhika-demo
Type 'clear' to erase memory.
Type 'exit' to stop.

You: exit
Chat ended.


In [3]:
# ============================================================
# VIEW CURRENT SESSION MEMORY
# ============================================================

history = get_session("radhika-demo")

print("=" * 60)
print("CURRENT CONVERSATION MEMORY")
print("=" * 60)

for i, message in enumerate(history.messages, 1):

    print(f"\n{i}. {message['role'].upper()}")
    print(message["content"])

print("\nTotal stored messages:", len(history.messages))
print("Total turns:", len(history.messages) // 2)

CURRENT CONVERSATION MEMORY

Total stored messages: 0
Total turns: 0


In [4]:
# ============================================================
# REQUIRED FIVE-TURN TEST
# ============================================================

session_id = "assignment-test"

test_messages = [
    "My name is Radhika.",
    "I am building a healthcare AI assistant using Python.",
    "What am I building?",
    "It should help patients understand their symptoms.",
    "What should it help them understand?"
]

for turn, message in enumerate(test_messages, 1):

    response = chat_with_memory(
        session_id,
        message
    )

    print("\n" + "=" * 70)
    print(f"TURN {turn}")
    print("USER:", message)
    print("AI:", response)


TURN 1
USER: My name is Radhika.
AI: Hello Radhika! How can I assist you today?

TURN 2
USER: I am building a healthcare AI assistant using Python.
AI: Hello Radhika! That sounds like an exciting project for your healthcare AI assistant. To get started, could you tell me more about what specific aspects of AI health care you'd like to integrate into this assistant? For example, do you have particular needs in terms of data handling, user interfaces, or any other features that you're interested in incorporating?

Additionally, if you're planning to use Python as the primary language for development, could you provide some details on how you plan to set up your environment, including any necessary tools or libraries? This will help us tailor our advice accordingly.

Let's dive deeper into these areas to see how we can best support your AI health care project.

TURN 3
USER: What am I building?
AI: Hello Radhika! You are building a healthcare AI assistant using Python. It sounds like an i

In [5]:
# ============================================================
# MEMORY WINDOW TRUNCATION TEST
# ============================================================

session_id = "truncation-test"

for i in range(12):

    message = (
        f"This is turn {i + 1}. "
        f"Remember that my project fact is "
        f"ProjectFact-{i + 1}."
    )

    response = chat_with_memory(
        session_id,
        message
    )

    history = get_session(session_id)

    print(
        f"Turn {i + 1:2d} | "
        f"Stored messages: {len(history.messages)}"
    )

Turn  1 | Stored messages: 2
Turn  2 | Stored messages: 4
Turn  3 | Stored messages: 6
Turn  4 | Stored messages: 8
Turn  5 | Stored messages: 10
Turn  6 | Stored messages: 12
Turn  7 | Stored messages: 14
Turn  8 | Stored messages: 16
Turn  9 | Stored messages: 18
Turn 10 | Stored messages: 20
Turn 11 | Stored messages: 13
Turn 12 | Stored messages: 15


In [6]:
# ============================================================
# TEST FASTAPI /chat ENDPOINT
# ============================================================

import requests

URL = "http://127.0.0.1:8000/chat"

session_id = "api-session-001"

messages = [
    "My name is Radhika.",
    "I am learning Python and AI.",
    "What am I learning?",
    "I want to build a healthcare assistant.",
    "What do I want to build?"
]

for i, message in enumerate(messages, 1):

    response = requests.post(
        URL,
        json={
            "session_id": session_id,
            "message": message
        }
    )

    print("=" * 70)
    print(f"TURN {i}")
    print("USER:", message)

    if response.status_code == 200:
        data = response.json()

        print("AI:", data["response"])
        print("SESSION ID:", data["session_id"])
        print("STORED MESSAGES:", data["stored_messages"])

    else:
        print("ERROR:", response.status_code)
        print(response.text)

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x79b000ce86e0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [7]:
# ============================================================
# CHECK WHETHER FASTAPI IS RUNNING
# ============================================================

import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

result = sock.connect_ex(("127.0.0.1", 8000))

sock.close()

if result == 0:
    print("✅ FastAPI is running on port 8000")
else:
    print("❌ FastAPI is NOT running on port 8000")

❌ FastAPI is NOT running on port 8000


In [8]:
# ============================================================
# START FASTAPI SERVER
# ============================================================

import uvicorn
import threading
import time

def start_fastapi():

    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=start_fastapi,
    daemon=True
)

server_thread.start()

# Give Uvicorn time to start
time.sleep(3)

print("Checking server...")

import socket

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

result = sock.connect_ex(("127.0.0.1", 8000))

sock.close()

if result == 0:
    print("✅ FastAPI successfully started!")
    print("http://127.0.0.1:8000")
else:
    print("❌ FastAPI failed to start.")

Exception in thread Thread-6 (start_fastapi):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_542/3863544559.py", line 11, in start_fastapi
    uvicorn.run(
    ~~~~~~~~~~~^
        app,
        ^^^^
    ...<2 lines>...
        log_level="info"
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/main.py", line 621, in run
    server.run()
    ~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 77, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argument 'loop_factory'


Checking server...
❌ FastAPI failed to start.


In [9]:
# ============================================================
# FIX UVICORN / NEST_ASYNCIO COMPATIBILITY
# ============================================================

!pip install -q "uvicorn==0.30.6"

print("Uvicorn version fixed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires uvicorn<1,>=0.34, but you have uvicorn 0.30.6 which is incompatible.
Uvicorn version fixed.


In [10]:
!pip install -q -U "uvicorn>=0.34,<1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 2.0 MB/s eta 0:00:00


In [11]:
import uvicorn
print("Uvicorn:", uvicorn.__version__)

Uvicorn: 0.52.4


In [12]:
# Remove nest_asyncio patching
try:
    import nest_asyncio
    print("nest_asyncio imported, but we will NOT apply it.")
except:
    pass

nest_asyncio imported, but we will NOT apply it.


In [13]:
import uvicorn
import threading
import time

def start_fastapi():
    uvicorn.run(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=start_fastapi,
    daemon=True
)

server_thread.start()

time.sleep(3)

print("Checking FastAPI...")

Exception in thread Thread-7 (start_fastapi):
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_542/301899607.py", line 6, in start_fastapi
    uvicorn.run(
    ~~~~~~~~~~~^
        app,
        ^^^^
    ...<2 lines>...
        log_level="info"
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/main.py", line 621, in run
    server.run()
    ~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 77, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
TypeError: _patch_asyncio.<locals>.run() got an unexpected keyword argument 'loop_factory'


Checking FastAPI...


In [14]:
# ============================================================
# FASTAPI SERVER - COLAB SAFE VERSION
# ============================================================

import os
import textwrap

server_code = r'''
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional
import uuid

app = FastAPI(
    title="Stateful AI Conversation API",
    version="1.0"
)

# ------------------------------------------------------------
# Conversation Memory
# ------------------------------------------------------------

class ConversationHistory:

    def __init__(self):
        self.messages = []

    def append(self, role, content):
        self.messages.append({
            "role": role,
            "content": content
        })

    def get_context(self, max_turns=5):
        return self.messages[-(max_turns * 2):]

    def clear(self):
        self.messages = []

# ------------------------------------------------------------
# Sessions
# ------------------------------------------------------------

sessions = {}

def get_session(session_id):

    if session_id not in sessions:
        sessions[session_id] = ConversationHistory()

    return sessions[session_id]

# ------------------------------------------------------------
# Request model
# ------------------------------------------------------------

class ChatRequest(BaseModel):

    message: str
    session_id: Optional[str] = None

# ------------------------------------------------------------
# Demo response
# ------------------------------------------------------------

def generate_response(history, message):

    context = history.get_context(5)

    # For now, return the conversation context.
    # We will connect the local LLM after the API works.
    if context:

        previous = []

        for item in context:
            previous.append(
                f"{item['role']}: {item['content']}"
            )

        return (
            "I remember our conversation.\n\n"
            "Previous context:\n"
            + "\n".join(previous)
            + f"\n\nCurrent message: {message}"
        )

    return f"You said: {message}"

# ------------------------------------------------------------
# Chat endpoint
# ------------------------------------------------------------

@app.post("/chat")
def chat(request: ChatRequest):

    session_id = request.session_id

    if not session_id:
        session_id = str(uuid.uuid4())

    history = get_session(session_id)

    response = generate_response(
        history,
        request.message
    )

    history.append(
        "user",
        request.message
    )

    history.append(
        "assistant",
        response
    )

    return {
        "session_id": session_id,
        "response": response,
        "stored_messages": len(history.messages)
    }

# ------------------------------------------------------------
# Clear session
# ------------------------------------------------------------

@app.delete("/session/{session_id}")
def clear_session(session_id: str):

    if session_id in sessions:

        sessions[session_id].clear()

        return {
            "status": "success",
            "message": "Memory cleared"
        }

    return {
        "status": "not_found"
    }
'''

with open("/content/server.py", "w") as f:
    f.write(server_code)

print("✅ server.py created")

✅ server.py created


In [15]:
# ============================================================
# START FASTAPI AS SEPARATE PROCESS
# ============================================================

import subprocess
import time
import requests

server_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "server:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="/content"
)

time.sleep(3)

try:

    response = requests.get(
        "http://127.0.0.1:8000/docs",
        timeout=5
    )

    print("Status:", response.status_code)

    if response.status_code == 200:
        print("✅ FastAPI is running!")
    else:
        print("⚠️ Server responded:", response.status_code)

except Exception as e:

    print("❌ Server failed:")
    print(e)

❌ Server failed:
HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /docs (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x79b0010df9d0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [16]:
# ============================================================
# TEST FASTAPI
# ============================================================

import requests

URL = "http://127.0.0.1:8000/chat"

session_id = "radhika-001"

messages = [
    "My name is Radhika.",
    "I am learning Python.",
    "What am I learning?"
]

for turn, message in enumerate(messages, 1):

    response = requests.post(
        URL,
        json={
            "session_id": session_id,
            "message": message
        }
    )

    print("=" * 60)
    print("TURN:", turn)
    print("USER:", message)
    print("STATUS:", response.status_code)
    print("AI:", response.json()["response"])

TURN: 1
USER: My name is Radhika.
STATUS: 200
AI: You said: My name is Radhika.
TURN: 2
USER: I am learning Python.
STATUS: 200
AI: I remember our conversation.

Previous context:
user: My name is Radhika.
assistant: You said: My name is Radhika.

Current message: I am learning Python.
TURN: 3
USER: What am I learning?
STATUS: 200
AI: I remember our conversation.

Previous context:
user: My name is Radhika.
assistant: You said: My name is Radhika.
user: I am learning Python.
assistant: I remember our conversation.

Previous context:
user: My name is Radhika.
assistant: You said: My name is Radhika.

Current message: I am learning Python.

Current message: What am I learning?


In [17]:
# ============================================================
# STATEFUL AI ASSISTANT — FINAL FASTAPI SERVER
# Local Qwen LLM + Session Memory
# ============================================================

# IMPORTANT:
# This cell creates a complete server.py file.
# It does NOT use OpenAI API.
# It does NOT use nest_asyncio.

server_code = r'''
import torch
import uuid

from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)


# ============================================================
# MODEL
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading local model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

print("Model loaded!")


# ============================================================
# LLM FUNCTION
# ============================================================

def generate_response(prompt, max_new_tokens=180):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


# ============================================================
# CONVERSATION HISTORY
# ============================================================

class ConversationHistory:

    def __init__(self):

        self.messages = []

    def append(self, role, content):

        self.messages.append({
            "role": role,
            "content": content
        })

    def get_context(self, max_turns=5):

        return self.messages[-(max_turns * 2):]

    def clear(self):

        self.messages = []


# ============================================================
# SESSION STORAGE
# ============================================================

sessions = {}


def get_session(session_id):

    if session_id not in sessions:

        sessions[session_id] = ConversationHistory()

    return sessions[session_id]


# ============================================================
# CONTEXT BUILDER
# ============================================================

def build_prompt(history, current_message):

    context = history.get_context(5)

    context_text = ""

    for message in context:

        context_text += (
            f"{message['role'].upper()}: "
            f"{message['content']}\n"
        )

    return f"""
You are a helpful AI assistant.

Use the conversation history to answer
the current user's question.

CONVERSATION HISTORY:

{context_text}

CURRENT USER MESSAGE:

USER: {current_message}

Answer naturally and use previous context
when relevant.
"""


# ============================================================
# FASTAPI
# ============================================================

app = FastAPI(
    title="Stateful AI Conversation API",
    version="1.0"
)


# ============================================================
# REQUEST MODEL
# ============================================================

class ChatRequest(BaseModel):

    message: str

    session_id: Optional[str] = None


# ============================================================
# CHAT ENDPOINT
# ============================================================

@app.post("/chat")
def chat(request: ChatRequest):

    session_id = request.session_id

    if not session_id:

        session_id = str(uuid.uuid4())

    history = get_session(
        session_id
    )

    prompt = build_prompt(
        history,
        request.message
    )

    response = generate_response(
        prompt
    )

    history.append(
        "user",
        request.message
    )

    history.append(
        "assistant",
        response
    )

    return {
        "session_id": session_id,
        "response": response,
        "stored_messages": len(history.messages),
        "memory_window": "last 5 turns"
    }


# ============================================================
# CLEAR MEMORY
# ============================================================

@app.delete("/session/{session_id}")
def clear_session(session_id: str):

    if session_id in sessions:

        sessions[session_id].clear()

        return {
            "status": "success",
            "message": "Conversation memory cleared"
        }

    return {
        "status": "not_found",
        "message": "Session does not exist"
    }


# ============================================================
# HEALTH CHECK
# ============================================================

@app.get("/")
def root():

    return {
        "status": "online",
        "project": "Stateful AI Conversation Assistant"
    }
'''

with open("/content/server.py", "w") as f:
    f.write(server_code)

print("✅ Final AI server created.")

✅ Final AI server created.


In [18]:
# Stop previous FastAPI process

try:
    server_process.terminate()
    server_process.wait(timeout=5)
    print("✅ Old server stopped.")
except:
    print("No previous server process found.")

✅ Old server stopped.


In [19]:
import subprocess
import time
import requests

server_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "server:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="/content"
)

print("Loading Qwen model and starting FastAPI...")

time.sleep(8)

try:

    response = requests.get(
        "http://127.0.0.1:8000/",
        timeout=5
    )

    print("Status:", response.status_code)
    print(response.json())

except Exception as e:

    print("Server error:", e)

Loading Qwen model and starting FastAPI...
Server error: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x79b0010b7890>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [20]:
import requests

URL = "http://127.0.0.1:8000/chat"

session_id = "radhika-final-test"

messages = [
    "My name is Radhika.",
    "I am learning Python and artificial intelligence.",
    "What am I learning?",
    "I want to build a healthcare AI assistant.",
    "What do I want to build?"
]

for turn, message in enumerate(messages, 1):

    response = requests.post(
        URL,
        json={
            "session_id": session_id,
            "message": message
        }
    )

    data = response.json()

    print("=" * 70)
    print(f"TURN {turn}")
    print("USER:", message)
    print("AI:", data["response"])
    print("SESSION:", data["session_id"])
    print("STORED MESSAGES:", data["stored_messages"])

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /chat (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x79b001132690>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [21]:
# STOP ALL OLD UVICORN PROCESSES

import subprocess

subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True
)

print("Old Uvicorn processes stopped.")

Old Uvicorn processes stopped.


In [22]:
import os

print("server.py exists:", os.path.exists("/content/server.py"))

if os.path.exists("/content/server.py"):
    print("Size:", os.path.getsize("/content/server.py"), "bytes")

server.py exists: True
Size: 5116 bytes


In [23]:
import subprocess
import time
import requests

server_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "server:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(10)

# Check whether process died
if server_process.poll() is not None:

    print("❌ SERVER STOPPED")

    output = server_process.stdout.read()

    print("\nSERVER ERROR:")
    print(output)

else:

    print("✅ Server process is still running")

    try:

        response = requests.get(
            "http://127.0.0.1:8000/",
            timeout=5
        )

        print("HTTP STATUS:", response.status_code)
        print("RESPONSE:", response.json())

    except Exception as e:

        print("❌ Server exists but HTTP connection failed:")
        print(e)

✅ Server process is still running
❌ Server exists but HTTP connection failed:
HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x79b001132be0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [24]:
# ============================================================
# MINIMAL FASTAPI TEST
# ============================================================

minimal_server = """
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {
        "status": "working",
        "message": "FastAPI is running"
    }

@app.post("/chat")
def chat(data: dict):
    return {
        "response": "FastAPI received your message",
        "session_id": data.get("session_id")
    }
"""

with open("/content/test_server.py", "w") as f:
    f.write(minimal_server)

print("Minimal server created.")

Minimal server created.


In [25]:
# Start minimal server

import subprocess
import time
import requests

test_process = subprocess.Popen(
    [
        "python",
        "-m",
        "uvicorn",
        "test_server:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(3)

try:

    r = requests.get(
        "http://127.0.0.1:8000/",
        timeout=5
    )

    print("STATUS:", r.status_code)
    print(r.json())

except Exception as e:

    print("ERROR:", e)

    if test_process.poll() is not None:
        print("\nSERVER LOG:")
        print(test_process.stdout.read())

STATUS: 200
{'status': 'online', 'project': 'Stateful AI Conversation Assistant'}


In [26]:
# ============================================================
# DAY 23 — MEMORY WINDOW TRUNCATION
# ============================================================
# This cell:
# 1. Creates ConversationHistory
# 2. Keeps the latest 5 turns
# 3. Summarizes oldest 5 turns after 10 turns
# 4. Tests the complete process
# ============================================================

# ------------------------------------------------------------
# 1. ConversationHistory
# ------------------------------------------------------------

class ConversationHistory:

    def __init__(self):
        self.messages = []

    def append(self, role, content):

        self.messages.append({
            "role": role,
            "content": content
        })

    def get_context(self, max_turns=5):

        max_messages = max_turns * 2

        return self.messages[-max_messages:]

    def clear(self):

        self.messages = []

    def show(self):

        for i, message in enumerate(self.messages, 1):

            print(
                f"{i:02d}. "
                f"{message['role'].upper()}: "
                f"{message['content']}"
            )


# ------------------------------------------------------------
# 2. LLM summarization
# ------------------------------------------------------------

def summarize_oldest_turns(messages):

    conversation = ""

    for message in messages:

        conversation += (
            f"{message['role'].upper()}: "
            f"{message['content']}\n"
        )

    prompt = f"""
You are a conversation memory system.

Summarize the following old conversation.

Keep only information that could be useful
for answering future questions.

Preserve:
- names
- projects
- important facts
- preferences
- decisions
- important context

Do NOT invent anything.

OLD CONVERSATION:

{conversation}

Return a short memory summary.
"""

    return generate_response(
        prompt,
        max_new_tokens=120
    )


# ------------------------------------------------------------
# 3. Compress memory
# ------------------------------------------------------------

def compress_memory(history):

    total_turns = len(history.messages) // 2

    # Nothing to compress yet
    if total_turns <= 10:
        return False

    print("\n🔄 Memory limit reached!")
    print("Summarizing oldest 5 turns...")

    # First 5 turns = 10 messages
    oldest_five_turns = history.messages[:10]

    summary = summarize_oldest_turns(
        oldest_five_turns
    )

    # Keep everything after the oldest 5 turns
    remaining = history.messages[10:]

    # Replace old conversation with summary
    history.messages = [
        {
            "role": "system",
            "content": (
                "MEMORY SUMMARY:\n" + summary
            )
        }
    ] + remaining

    print("✅ Oldest 5 turns summarized.")

    return True


# ------------------------------------------------------------
# 4. Create test conversation
# ------------------------------------------------------------

memory = ConversationHistory()


# ------------------------------------------------------------
# 5. Add 12 turns
# ------------------------------------------------------------

for turn in range(1, 13):

    memory.append(
        "user",
        f"My conversation turn is {turn}. "
        f"My project fact is Project-{turn}."
    )

    memory.append(
        "assistant",
        f"I understand the information from turn {turn}."
    )

    compress_memory(memory)

    print(
        f"Turn {turn:02d} → "
        f"Stored messages: {len(memory.messages)}"
    )


# ------------------------------------------------------------
# 6. Display final memory
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("FINAL MEMORY")
print("=" * 70)

memory.show()


# ------------------------------------------------------------
# 7. Show current 5-turn context
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("CURRENT 5-TURN CONTEXT")
print("=" * 70)

context = memory.get_context(5)

for message in context:

    print(
        f"{message['role'].upper()}: "
        f"{message['content']}"
    )


# ------------------------------------------------------------
# 8. Final result
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("MEMORY TRUNCATION TEST COMPLETE")
print("=" * 70)

print("""
✓ ConversationHistory created
✓ append() implemented
✓ get_context() implemented
✓ clear() implemented
✓ Maximum active conversation window = 5 turns
✓ Compression triggered after 10 turns
✓ Oldest 5 turns summarized using local LLM
✓ Summary replaces old messages
""")

Turn 01 → Stored messages: 2
Turn 02 → Stored messages: 4
Turn 03 → Stored messages: 6
Turn 04 → Stored messages: 8
Turn 05 → Stored messages: 10
Turn 06 → Stored messages: 12
Turn 07 → Stored messages: 14
Turn 08 → Stored messages: 16
Turn 09 → Stored messages: 18
Turn 10 → Stored messages: 20

🔄 Memory limit reached!
Summarizing oldest 5 turns...
✅ Oldest 5 turns summarized.
Turn 11 → Stored messages: 13
Turn 12 → Stored messages: 15


FINAL MEMORY
01. SYSTEM: MEMORY SUMMARY:
The user has provided a series of project facts in chronological order. The last fact mentioned is Project-5, which indicates that there have been no further project facts provided since the start of their conversation.
02. USER: My conversation turn is 6. My project fact is Project-6.
03. ASSISTANT: I understand the information from turn 6.
04. USER: My conversation turn is 7. My project fact is Project-7.
05. ASSISTANT: I understand the information from turn 7.
06. USER: My conversation turn is 8. My project f

In [27]:
# ============================================================
# DAY 23 — MEMORY VS NO MEMORY + TOKEN ANALYSIS
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Five-turn experiment
# ------------------------------------------------------------

test_messages = [
    "My name is Radhika.",
    "I am building a healthcare AI assistant using Python.",
    "What am I building?",
    "It should help patients understand their symptoms.",
    "What should it help them understand?"
]


# ------------------------------------------------------------
# 2. WITH MEMORY
# ------------------------------------------------------------

print("=" * 80)
print("EXPERIMENT 1 — WITH MEMORY")
print("=" * 80)

memory_test = ConversationHistory()

with_memory_results = []

for turn, message in enumerate(test_messages, 1):

    prompt = build_prompt(
        memory_test,
        message
    )

    answer = generate_response(
        prompt,
        max_new_tokens=120
    )

    memory_test.append(
        "user",
        message
    )

    memory_test.append(
        "assistant",
        answer
    )

    with_memory_results.append(answer)

    print(f"\nTURN {turn}")
    print("USER:", message)
    print("AI:", answer)


# ------------------------------------------------------------
# 3. WITHOUT MEMORY
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("EXPERIMENT 2 — WITHOUT MEMORY")
print("=" * 80)

without_memory_results = []

for turn, message in enumerate(test_messages, 1):

    prompt = f"""
You are a helpful AI assistant.

Answer this user message independently.
You do NOT have access to previous messages.

USER:
{message}
"""

    answer = generate_response(
        prompt,
        max_new_tokens=120
    )

    without_memory_results.append(answer)

    print(f"\nTURN {turn}")
    print("USER:", message)
    print("AI:", answer)


# ------------------------------------------------------------
# 4. Display comparison
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("MEMORY COMPARISON")
print("=" * 80)

comparison = pd.DataFrame({
    "Turn": range(1, 6),
    "User Message": test_messages,
    "With Memory": with_memory_results,
    "Without Memory": without_memory_results
})

display(comparison)


# ------------------------------------------------------------
# 5. Token counting
# ------------------------------------------------------------

def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )


def context_token_count(history):

    context = history.get_context(5)

    total = 0

    for message in context:

        total += count_tokens(
            message["content"]
        )

    return total


# ------------------------------------------------------------
# 6. Calculate tokens for each turn
# ------------------------------------------------------------

token_memory = ConversationHistory()

token_rows = []

for turn, message in enumerate(test_messages, 1):

    context = token_memory.get_context(5)

    context_tokens = sum(
        count_tokens(m["content"])
        for m in context
    )

    current_message_tokens = count_tokens(
        message
    )

    token_rows.append({
        "Turn": turn,
        "Current Message Tokens":
            current_message_tokens,
        "Memory Context Tokens":
            context_tokens,
        "Total Input Tokens":
            current_message_tokens + context_tokens
    })

    # Add fake assistant response so that
    # the memory grows for the experiment.
    token_memory.append(
        "user",
        message
    )

    token_memory.append(
        "assistant",
        with_memory_results[turn - 1]
    )


token_df = pd.DataFrame(token_rows)

print("\n")
print("=" * 80)
print("TOKEN USAGE PER TURN")
print("=" * 80)

display(token_df)


# ------------------------------------------------------------
# 7. Calculate average memory overhead
# ------------------------------------------------------------

average_memory_tokens = token_df[
    "Memory Context Tokens"
].mean()

average_input_tokens = token_df[
    "Total Input Tokens"
].mean()

print(
    "Average additional memory tokens per turn:",
    round(average_memory_tokens, 2)
)

print(
    "Average total input tokens per turn:",
    round(average_input_tokens, 2)
)


# ------------------------------------------------------------
# 8. Monthly workload estimate
# ------------------------------------------------------------

DAILY_USERS = 1000
TURNS_PER_SESSION = 10
DAYS_PER_MONTH = 30

# Use the measured average memory tokens.
memory_tokens_per_turn = average_memory_tokens

additional_tokens_daily = (
    DAILY_USERS
    * TURNS_PER_SESSION
    * memory_tokens_per_turn
)

additional_tokens_monthly = (
    additional_tokens_daily
    * DAYS_PER_MONTH
)

print("\n")
print("=" * 80)
print("MONTHLY MEMORY WORKLOAD")
print("=" * 80)

print(
    "Daily users:",
    DAILY_USERS
)

print(
    "Turns per session:",
    TURNS_PER_SESSION
)

print(
    "Average memory tokens/turn:",
    round(memory_tokens_per_turn, 2)
)

print(
    "Additional tokens/day:",
    round(additional_tokens_daily)
)

print(
    "Additional tokens/month:",
    round(additional_tokens_monthly)
)


# ------------------------------------------------------------
# 9. Local model cost
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("COST ANALYSIS")
print("=" * 80)

print("""
This project uses a local Hugging Face model.

Therefore:

External API cost = $0
OpenAI API cost   = $0

However, memory still increases:

• GPU computation
• VRAM usage
• inference latency
• processing time

The 5-turn window prevents unlimited context growth.
The 10-turn compression mechanism further controls
memory size by summarizing older conversation history.
""")


# ------------------------------------------------------------
# 10. Final experiment summary
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)

print("""
MEMORY SYSTEM
--------------
✓ Session-specific conversation state
✓ Last 5 turns included in prompts
✓ Oldest 5 turns summarized after 10 turns
✓ ConversationHistory.append()
✓ ConversationHistory.get_context()
✓ ConversationHistory.clear()

EXPERIMENT
----------
✓ Five-turn conversation tested
✓ Pronoun/context-dependent follow-ups tested
✓ With-memory responses generated
✓ Without-memory responses generated

TOKEN ANALYSIS
--------------
✓ Context tokens measured
✓ Average memory overhead calculated
✓ Monthly workload estimated

API
---
✓ FastAPI architecture implemented
✓ /chat endpoint implemented
✓ Session ID support implemented
✓ Session clearing implemented
""")

EXPERIMENT 1 — WITH MEMORY

TURN 1
USER: My name is Radhika.
AI: Hello Radhika! How can I assist you today?

TURN 2
USER: I am building a healthcare AI assistant using Python.
AI: Hello Radhika! That sounds like an exciting project for your healthcare AI assistant. What specifically do you need help with in terms of integrating Python into your healthcare application? Do you have any particular aspects or challenges you're facing that you'd like to discuss further?

TURN 3
USER: What am I building?
AI: Hello Radhika! You've asked me what you're building. While I don't have access to specific details about the healthcare AI assistant, I can provide some general insights based on common projects people might be working on. If you could describe more about the healthcare AI assistant you're planning, I could offer more tailored advice.

TURN 4
USER: It should help patients understand their symptoms.
AI: Hello Radhika! Great to hear that you're thinking about building a healthcare AI assis

,Turn,User Message,With Memory,Without Memory
0,1,My name is Radhika.,Hello Radhika! How can I assist you today?,Hello Radhika! How can I assist you today?
1,2,I am building a healthcare AI assistant using ...,Hello Radhika! That sounds like an exciting pr...,That sounds like an exciting project! Building...
2,3,What am I building?,Hello Radhika! You've asked me what you're bui...,"Based on the provided user message, it is not ..."
3,4,It should help patients understand their sympt...,Hello Radhika! Great to hear that you're think...,Understood! Here's how I can assist:\n\n1. **U...
4,5,What should it help them understand?,Hello Radhika! Your healthcare AI assistant sh...,When interacting with users on platforms like ...




TOKEN USAGE PER TURN


,Turn,Current Message Tokens,Memory Context Tokens,Total Input Tokens
0,1,7,0,7
1,2,10,19,29
2,3,5,82,87
3,4,8,153,161
4,5,7,281,288


Average additional memory tokens per turn: 107.0
Average total input tokens per turn: 114.4


MONTHLY MEMORY WORKLOAD
Daily users: 1000
Turns per session: 10
Average memory tokens/turn: 107.0
Additional tokens/day: 1070000
Additional tokens/month: 32100000


COST ANALYSIS

This project uses a local Hugging Face model.

Therefore:

External API cost = $0
OpenAI API cost   = $0

However, memory still increases:

• GPU computation
• VRAM usage
• inference latency
• processing time

The 5-turn window prevents unlimited context growth.
The 10-turn compression mechanism further controls
memory size by summarizing older conversation history.



FINAL RESULTS

MEMORY SYSTEM
--------------
✓ Session-specific conversation state
✓ Last 5 turns included in prompts
✓ Oldest 5 turns summarized after 10 turns
✓ ConversationHistory.append()
✓ ConversationHistory.get_context()
✓ ConversationHistory.clear()

EXPERIMENT
----------
✓ Five-turn conversation tested
✓ Pronoun/context-dependent follow-ups tes